# Prefix GCN — Sliding-Window Classification on BPI Challenge 2012

This notebook applies the **Prefix GCN** model to the BPI 2012 dataset.

BPI 2012 has longer cases (median ~20 events), so the `prefix_size` parameter
can be set higher to give the model more context.

### Notebook outline
1. Load & Explore
2. Train
3. Evaluate
4. Summary

In [1]:
from pathlib import Path
import sys, warnings

root = Path.cwd().resolve()
src = root / "src" if (root / "src").exists() else root.parent / "src"
sys.path.insert(0, str(src))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load & Explore the Dataset

In [2]:
import pandas as pd

data_dir = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd() / "examples" / "data"
df = pd.read_csv(data_dir / "BPI_Challenge_2012.csv")

df = df.rename(columns={
    "case:concept:name": "case_id",
    "concept:name":      "event",
    "time:timestamp":    "time",
    "lifecycle:transition": "status",
    "case:AMOUNT_REQ":      "amount_req",
})
df = df.dropna(subset=["case_id", "event", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
df = df[df.groupby("case_id")["case_id"].transform("size") >= 2].reset_index(drop=True)

lengths = df.groupby("case_id").size()
print(f"Cases: {len(lengths)}  |  Events: {len(df)}")
print(f"Sequence lengths — min: {lengths.min()}, median: {lengths.median():.0f}, max: {lengths.max()}")
df.head()

Cases: 13087  |  Events: 262200
Sequence lengths — min: 3, median: 11, max: 175


,org:resource,status,event,time,case:REG_DATE,case_id,amount_req
0,112.0,COMPLETE,A_SUBMITTED,2011-10-01 00:38:44.546000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
1,112.0,COMPLETE,A_PARTLYSUBMITTED,2011-10-01 00:38:44.880000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
2,112.0,COMPLETE,A_PREACCEPTED,2011-10-01 00:39:37.906000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
3,112.0,SCHEDULE,W_Completeren aanvraag,2011-10-01 00:39:38.875000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
4,NaN,START,W_Completeren aanvraag,2011-10-01 11:36:46.437000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000


## 2. Train the Model

We use `prefix_size=10` here (larger window than Helpdesk)
since BPI 2012 cases are longer on average.

In [3]:
from timegnn import prefix_gcn

result = prefix_gcn(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    cat_event=["event"],
    num_event=[],
    cat_seq=[],
    num_seq=["amount_req"],
    prefix_size=10,
    num_epochs=3,
    batch_size=16,
    stratify=False,
)

model = result["model"]
pd.DataFrame(result["history"])

,epoch,train_loss,train_acc,test_loss,test_acc
0,1,0.740773,0.758998,0.606022,0.798519
1,2,0.599854,0.796900,0.588666,0.790795
2,3,0.580749,0.801826,0.569752,0.804291


## 3. Evaluate with Top-k Metrics

In [4]:
import torch
from torch.utils.data import DataLoader
from timegnn import top_k_accuracy
from timegnn.data import PrefixGCNTransformer
from timegnn.data.pyg import PrefixDataset, custom_collate_prefix

PREFIX_SIZE = 10

df_prefix = df[df.groupby("case_id")["case_id"].transform("size") >= PREFIX_SIZE].reset_index(drop=True)

transformer = PrefixGCNTransformer(
    case_col="case_id", event_col="event", time_col="time",
    prefix_size=PREFIX_SIZE,
    cat_event=["event"], num_event=[],
    cat_seq=[], num_seq=["amount_req"],
)
transformed = transformer.transform(df_prefix)

dataset = PrefixDataset(transformed.event_features, transformed.sequence_features, transformed.labels)
loader  = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_prefix)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device).eval()

all_outputs, all_labels = [], []
with torch.no_grad():
    for batch_event, batch_seq, batch_labels in loader:
        out = model(batch_event.to(device), batch_seq.to(device))
        all_outputs.append(out.cpu())
        all_labels.append(batch_labels.cpu())

outputs = torch.cat(all_outputs)
labels  = torch.cat(all_labels)

print(f"Top-1 accuracy:  {top_k_accuracy(outputs, labels, k=1):.4f}")
print(f"Top-3 accuracy:  {top_k_accuracy(outputs, labels, k=3):.4f}")
print(f"Top-5 accuracy:  {top_k_accuracy(outputs, labels, k=5):.4f}")
print(f"\nTotal prefix samples: {len(labels)}")

Top-1 accuracy:  0.8041
Top-3 accuracy:  0.9707
Top-5 accuracy:  0.9893

Total prefix samples: 171529


## 4. Summary

On BPI 2012, the larger `prefix_size=10` gives the model more context per prefix.
The loan amount (`amount_req`) serves as a useful sequence-level feature:
different loan sizes may follow different process paths.

Try experimenting with:
- `prefix_size` (5, 10, 15) — trade-off between context and data coverage
- `stratify=True` — useful when class distribution is highly imbalanced